In [1]:
import pandas as pd
import numpy as np
import missingno as msno

In [2]:
df = pd.read_csv("../data/StudentsPerformance.csv") #Importación inicial del archivo

# Análisis de rendimiento académico de estudiantes - Big Data IDIA 222

Con el fin de probar la reproductibilidad de archivos de código python mediante la aplicación de conocimientos previos sobre el manejo de repositorios y clonación de entornos para su ejecución, se realizará un trabajo de análisis y procesamiento de datos con determinados parámetros (sobre librerías, enfáticamente) para su posterior reproducción en una máquina tercera.

In [3]:
#Despliegue inicial de CSV para verificar importación
df

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75
...,...,...,...,...,...,...,...,...
995,female,group E,master's degree,standard,completed,88,99,95
996,male,group C,high school,free/reduced,none,62,55,55
997,female,group C,high school,free/reduced,completed,59,71,65
998,female,group D,some college,standard,completed,68,78,77


## Exploración inicial

Lo primero siempre se tratará de conocer la base de datos con la que estamos trabajando, en este proceso se examinan detalles como:
- número de registros
- Número de columnas
- Número de variables
- Tipos de datos
- Valores nulos (o faltantes)
- Duplicados (ya sea registros o columnas)
- Estadística descrpitiva

In [4]:
print("Exploración inicial")
print("Número de registros: ", df.shape[0]*df.shape[1])
print("Número de columnas: ", df.shape[1])
print("Número de filas: ", df.shape[0])

Exploración inicial
Número de registros:  8000
Número de columnas:  8
Número de filas:  1000


In [7]:
#Para tipos de datos, esperando encontrar datos categóricos como "str" y numéricos como "int64"/"float64"
df.dtypes 

gender                           str
race/ethnicity                   str
parental level of education      str
lunch                            str
test preparation course          str
math score                     int64
reading score                  int64
writing score                  int64
dtype: object

In [8]:
#Para evaluación de datos nulos, se opta por realizar el análisis por columna y por porcentaje
df.isnull().mean()*100

gender                         0.0
race/ethnicity                 0.0
parental level of education    0.0
lunch                          0.0
test preparation course        0.0
math score                     0.0
reading score                  0.0
writing score                  0.0
dtype: float64

In [19]:
#Para la verificación de duplicados en ambas columnas y filas:
duplicados_filas = df[df.duplicated()]

print("Filas duplicadas:")
duplicados_filas

duplicados_columnas = df.columns[df.columns.duplicated()]

print("Columnas duplicadas:")

if len(duplicados_columnas) > 0:
    display(pd.DataFrame({"Columnas duplicadas": duplicados_columnas}))
else:
    display(pd.DataFrame())

Filas duplicadas:
Columnas duplicadas:


""


In [23]:
#Estadística descriptiva (variables categóricas)
df.describe(include="str")

,gender,race/ethnicity,parental level of education,lunch,test preparation course
count,1000,1000,1000,1000,1000
unique,2,5,6,2,2
top,female,group C,some college,standard,none
freq,518,319,226,645,642


In [24]:
#Estadística descriptiva (datos numéricos)
df.describe()

,math score,reading score,writing score
count,1000.00000,1000.000000,1000.000000
mean,66.08900,69.169000,68.054000
std,15.16308,14.600192,15.195657
min,0.00000,17.000000,10.000000
25%,57.00000,59.000000,57.750000
50%,66.00000,70.000000,69.000000
75%,77.00000,79.000000,79.000000
max,100.00000,100.000000,100.000000


De forma general podemos ver que el dataframe inicial con el que trabajamos tiene un buen estado en lo que concierne a duplicados y/o nulos, ya que de ninguno de los dos se encontraron registros que puedan afectar al uso del mismo. Con esto se puede comenzar a hacer evaluaciones e identificación de patrones, aunque la estadística descrpitiva ya revela, cuando menos, los patrones más elementales de comportamiento de los datos.

## Limpieza y preprocesamientos

Como comprobamos en el punto anterior, el dataframe inicial presenta un buen estado de salud considerando que no existen duplicados ni nulos; por otro lado, únicamente se hará una evaluación sobre valores posibles para identificar:
- En variables categóricas: todas las respuestas únicas que hay disponibles
- En variables numéricas: rangos de valores

Esto con el fin de corroborar si no existen datos "extraños" en el dataframe.

In [30]:
#Análisis de valores únicos en variables categóricas
col_cat = ["gender", "race/ethnicity", "parental level of education", "lunch", "test preparation course"]
for col in col_cat:
    print()
    print(f"Columna:  {col}")
    print(f"Total de observaciones: {df[col].count()}")
    tabla = pd.DataFrame({
        "Frecuencia": df[col].value_counts()
    })
    display(tabla)


Columna:  gender
Total de observaciones: 1000


,Frecuencia
gender,
female,518
male,482



Columna:  race/ethnicity
Total de observaciones: 1000


,Frecuencia
race/ethnicity,
group C,319
group D,262
group B,190
group E,140
group A,89



Columna:  parental level of education
Total de observaciones: 1000


,Frecuencia
parental level of education,
some college,226
associate's degree,222
high school,196
some high school,179
bachelor's degree,118
master's degree,59



Columna:  lunch
Total de observaciones: 1000


,Frecuencia
lunch,
standard,645
free/reduced,355



Columna:  test preparation course
Total de observaciones: 1000


,Frecuencia
test preparation course,
none,642
completed,358


In [28]:
#Análisis de rango de valores en variables numéricas.
col_num = ["math score", "reading score", "writing score"]
for col in col_num:
    print()
    print(f"Columna: {col}, mínimo: {df[col].min()}, Máximo: {df[col].max()}")


Columna: math score, mínimo: 0, Máximo: 100

Columna: reading score, mínimo: 17, Máximo: 100

Columna: writing score, mínimo: 10, Máximo: 100


Con estas observaciones podemos descatar si hay o no algun dato irregular o que no pertenezca a las respuestas esperadas del dataframe. A pesar de ello, no se encontró ningún dato irregular, por lo que no es necesario realizar limpieza ni procesos especiales al momento.